# MLOps Assignment — Sentiment Fine-Tuning (v1)

**Group 21 · PGD AI · IIT Jodhpur**  |  Model: `distilbert-base-uncased`  ·  Dataset: **SST-2**

This notebook is **experiment v1** (`learning_rate = 3e-5`). The only hyperparameter that differs from v2 is the learning rate, so the W&B comparison is meaningful.

> **Before running:** Accelerator → **GPU T4** (Settings panel), and add Kaggle Secrets `WANDB_API_KEY` and `HF_TOKEN` (Add-ons → Secrets).


## Install dependencies


In [ ]:
!pip install -q -U "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" \
    wandb evaluate scikit-learn huggingface_hub

## Step 1 — Load Secrets in Kaggle
Tokens are read from Kaggle Secrets — never hardcoded.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os, wandb
from huggingface_hub import login

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN)
wandb.login()

## Data preparation & normalisation (Task 2)
Load SST-2, normalise text, drop missing/duplicates, stratified 80/10/10 split, and write `id2label.json`.


In [ ]:
import re, json
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

SAMPLE_SIZE, SEED = 20000, 42
ws = re.compile(r'\s+')

ds = load_dataset('stanfordnlp/sst2', split='train')
label_names = ds.features['label'].names  # ['negative', 'positive']
df = pd.DataFrame({'text': ds['sentence'], 'label': ds['label']})
print('raw rows:', len(df), '| classes:', df['label'].value_counts().to_dict())

# clean: normalise whitespace + lowercase, drop missing/empty, dedup
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].map(lambda t: ws.sub(' ', str(t)).strip().lower())
df = df[df['text'].str.len() > 0].drop_duplicates(subset=['text', 'label'])
print('after cleaning:', len(df))

# stratified down-sample to keep training within free GPU limits
if len(df) > SAMPLE_SIZE:
    frac = SAMPLE_SIZE / len(df)
    df = df.groupby('label', group_keys=False).apply(
        lambda g: g.sample(frac=frac, random_state=SEED)).reset_index(drop=True)

id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in enumerate(label_names)}
with open('id2label.json', 'w') as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, indent=2)
print('id2label:', id2label)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)
dd = DatasetDict(train=Dataset.from_pandas(train_df, preserve_index=False),
                 validation=Dataset.from_pandas(val_df, preserve_index=False),
                 test=Dataset.from_pandas(test_df, preserve_index=False))
print(dd)

## Load tokenizer & model from Hugging Face (Task 3)
`distilbert-base-uncased` loaded with the correct number of labels from `id2label`.


In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(id2label), id2label=id2label, label2id=label2id)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

dd = dd.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Step 2 — Initialise a W&B Run
All hyperparameters for **v1** are logged in the run config.


In [ ]:
wandb.init(
    project='mlops-assignment3',
    name='run-v1',
    config={
        'model':         MODEL_NAME,
        'epochs':        3,
        'batch_size':    16,
        'learning_rate': 3e-5,
        'version':       'v1',
        'platform':      'Kaggle',
        'dataset':       'SST-2',
    })

## Step 3 — Training Arguments & Trainer
`report_to='wandb'` streams training loss, validation loss, accuracy and F1 every epoch.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='weighted')}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='wandb',
    run_name='run-v1',
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=dd['train'], eval_dataset=dd['validation'],
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

## Final held-out test metrics


In [ ]:
test_metrics = trainer.evaluate(dd['test'], metric_key_prefix='test')
print(test_metrics)
wandb.log(test_metrics)

## Push best model to Hugging Face Hub (Task 5)
Run this in the **better-performing** notebook only, so the public repo holds the best model.
It also logs the model URL into the W&B run summary.


In [ ]:
HF_REPO = 'Rajath-g25ait2079/distilbert-sst2-sentiment'
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
wandb.run.summary['huggingface_model'] = f'https://huggingface.co/{HF_REPO}'
print('Pushed to', f'https://huggingface.co/{HF_REPO}')

In [ ]:
wandb.finish()